# 0.0 Setup

In [41]:
import os
import gc
import json
import random
import requests
from tqdm import tqdm
from pathlib import Path
from collections import defaultdict

import re
import pickle
import importlib
import numpy as np
import pandas as pd
import openpyxl as opxl
import matplotlib.pyplot as plt
from typing import Optional, Dict, Any, List

import geopandas as gpd
from difflib import SequenceMatcher
from sklearn.impute import SimpleImputer, KNNImputer

# None means 'no limit'
pd.set_option('display.max_columns', None)

PROJECT_DIR = Path.cwd().parent
if 'modules' not in os.listdir(Path.cwd()):
    os.chdir(PROJECT_DIR)

DATA_DIR = 'data'
DATA_PUBLIC_DIR = 'data/public'
DATA_PRIVATE_DIR = 'data/private'
DATA_PROCESSED_DIR = 'data/processed'
OUTPUT_DIR = 'output'

# 1.0. Load data

In [2]:
# Load school information table for joining geographic information of schools
fpath = 'output/processed_project_bukas_school_information.parquet'
sch_info = pd.read_parquet(fpath)
print(sch_info.shape)

(61442, 20)


## 1.1. Coordinates data

In [3]:
%%time
fname = 'processed_public_school_coordinates.parquet'
outpath = os.path.join(OUTPUT_DIR, fname)
df_pub = pd.read_parquet(outpath)

# Join school information
df_pub = df_pub.merge(
    sch_info[['school_id','old_region','province']],
    on='school_id',
    how='left'
)
print(df_pub.shape)

(47821, 7)
CPU times: user 27.6 ms, sys: 9.8 ms, total: 37.4 ms
Wall time: 42.2 ms


In [4]:
%%time
fname = 'processed_private_school_coordinates.parquet'
outpath = os.path.join(OUTPUT_DIR, fname)
df_priv = pd.read_parquet(outpath)

# Join school information
df_priv = df_priv.merge(
    sch_info[['school_id','old_region','province']],
    on='school_id',
    how='left'
)
print(df_priv.shape)

(11831, 6)
CPU times: user 20.8 ms, sys: 4.75 ms, total: 25.5 ms
Wall time: 29.5 ms


### Inspect

In [5]:
display(df_pub)

,school_id,longitude,latitude,coord_valid,coord_missing,old_region,province
0,100001,120.614372,18.266860,True,False,Region I,ILOCOS NORTE
1,100002,120.609487,18.251272,True,False,Region I,ILOCOS NORTE
2,100003,120.616050,18.234670,True,False,Region I,ILOCOS NORTE
3,100004,120.587415,18.250121,True,False,Region I,ILOCOS NORTE
4,100005,120.641002,18.294093,True,False,Region I,ILOCOS NORTE
...,...,...,...,...,...,...,...
47816,320702,121.053254,14.457782,True,False,NCR,NCR FOURTH DISTRICT
47817,131785,125.886580,8.558120,True,False,CARAGA,AGUSAN DEL SUR
47818,137272,121.260711,14.611189,True,False,Region IV-A,RIZAL
47819,306294,123.152462,7.515867,True,False,Region IX,ZAMBOANGA DEL SUR


In [6]:
display(df_priv)

,school_id,longitude,latitude,coord_valid,old_region,province
0,400860,120.6149,17.58521,True,CAR,ABRA
1,406102,120.6145,17.59657,True,CAR,ABRA
2,406104,120.61364,17.59447,True,CAR,ABRA
3,406107,120.617597,17.596118,True,CAR,ABRA
4,406108,120.614023,17.596496,True,CAR,ABRA
...,...,...,...,...,...,...
11826,407378,122.86493,7.35855,True,Region IX,ZAMBOANGA SIBUGAY
11827,410936,122.5626,7.87866,True,Region IX,ZAMBOANGA SIBUGAY
11828,456516,122.5626,7.87866,True,Region IX,ZAMBOANGA SIBUGAY
11829,456520,122.5626,7.87866,True,Region IX,ZAMBOANGA SIBUGAY


### Prep All Coords

In [7]:
# =============================================================================                                                                                    
# 2.4. Prepare School Data                                                                                                                                         
# =============================================================================                                                                                    
                                                                                                                                                                 
# Define target regions                                                                                                                                            
target_regions = ['NCR', 'Region IV-A', 'Region III']                                                                                                                            
print(f"Target regions are: {', '.join(target_regions)}\n")

# Filter public schools to target regions                                                                                                                          
mask = (
    (df_pub['old_region'].isin(target_regions))
    & (df_pub['coord_valid'] == True)
)
df_pub_filtered = df_pub[mask].copy()                                                                                         
df_pub_filtered['sector'] = 'public'                                                                                                                               
                                                                                                                                                                 
# Filter private schools to target regions                                                                                                                         
mask = (
    (df_priv['old_region'].isin(target_regions))
    & (df_priv['coord_valid'] == True)
)
df_priv_filtered = df_priv[mask].copy()                                                                                      
df_priv_filtered['sector'] = 'private'                                                                                                                             

# To account for public & private schools not included due to invalid coordinates
mask = df_pub['old_region'].isin(target_regions)
df_pub_in_regions = df_pub[mask]
invalid_pubs_in_regions = df_pub_in_regions[df_pub_in_regions['coord_valid'] == False]

mask = df_priv['old_region'].isin(target_regions)
df_priv_in_regions = df_priv[mask]
invalid_privs_in_regions = df_priv_in_regions[df_priv_in_regions['coord_valid'] == False]

print(f"Public schools in target regions: {len(df_pub_filtered):,}")                                                                                               
print(f"Valid Private schools in target regions: {len(df_priv_filtered):,}")                                                                                             
                                                                                                                                                                 
# Combine public and private schools with essential columns only                                                                                                   
essential_cols = ['school_id', 'longitude', 'latitude', 'sector']                                                                                                  
                                                                                                                                                                 
schools_df = pd.concat([                                                                                                                                           
  df_pub_filtered[essential_cols],                                                                                                                               
  df_priv_filtered[essential_cols]                                                                                                                               
], ignore_index=True)                                                                                                                                              
                                                                                                                                                                 
# Ensure school_id is string                                                                                                                                       
schools_df['school_id'] = schools_df['school_id'].astype(str)                                                                                                      
                                                                                                                                                                 
# Drop any rows with missing coordinates                                                                                                                           
schools_df = schools_df.dropna(subset=['longitude', 'latitude'])                                                                                                   
                                                                                                                                                                 
print(f"\nCombined schools DataFrame:")                                                                                                                            
print(f"  Total schools: {len(schools_df):,}")                                                                                                                     
print(f"  Columns: {list(schools_df.columns)}")                                                                                                                    
display(schools_df.head())

# Notes on invalidated schools
pct_inv_pub = np.round((len(invalid_pubs_in_regions) / len(df_pub_in_regions)) * 100, 2)
pct_inv_priv = np.round((len(invalid_privs_in_regions) / len(df_priv_in_regions)) * 100, 2)
print(f"Count of PUBLIC schools with invalid coordinates {len(invalid_pubs_in_regions):,} ({pct_inv_pub:.2f}%) of {len(df_pub_in_regions):,}")
print(f"Count of PRIVATE schools with invalid coordinates {len(invalid_privs_in_regions):,} ({pct_inv_priv:.2f}%) of {len(df_priv_in_regions):,}")

Target regions are: NCR, Region IV-A, Region III

Public schools in target regions: 8,078
Valid Private schools in target regions: 5,053

Combined schools DataFrame:
  Total schools: 13,131
  Columns: ['school_id', 'longitude', 'latitude', 'sector']


,school_id,longitude,latitude,sector
0,105692,121.037638,15.797337,public
1,105693,120.99518,15.85671,public
2,105694,120.99229,15.81548,public
3,105695,120.99469,15.78601,public
4,105696,121.023395,15.784624,public


Count of PUBLIC schools with invalid coordinates 45 (0.55%) of 8,123
Count of PRIVATE schools with invalid coordinates 681 (11.88%) of 5,734


In [8]:
for col in ['longitude','latitude']:
    schools_df[col] = schools_df[col].astype(float)

for col in ['school_id','sector']:
    schools_df[col] = schools_df[col].astype(str)

# fpath = 'output/all_schools_coordinates_ncr_region4a_region3.parquet'
# schools_df.to_parquet(fpath)

# 2.0. Graph
Section 1.0 contains the datasets we need to build a graph of public and private schools. We will mainly use `OSRM` to implement the routing of our schools. The output of this graph is a N x N matrix where N is the count of all public and private schools.

For now, we will contain our graph newtork to only contain public and private schools from the National Capital Region, Region IV-A (or CALABARZON), and Region III.

In [18]:
# Test if OSRM works
response = requests.get("http://osrm:5000/route/v1/driving/121.0,14.5;121.1,14.6")                        
print(response.json())

{'code': 'Ok', 'routes': [{'legs': [{'steps': [], 'weight': 1759.2, 'summary': '', 'duration': 1746.9, 'distance': 20537.1}], 'weight_name': 'routability', 'geometry': '}aowA{xoaV_B~Lk]aRu{@gUqIfSsk@x@iQvEeF_MvAgQs|@|HyGa~@eKi\\m_CmzCiRuKuo@kNwfA{p@su@mRbXgb@sE_bBgWXmEyr@_a@sUoPsAiAgl@zJQlAzBfEkE', 'weight': 1759.2, 'duration': 1746.9, 'distance': 20537.1}], 'waypoints': [{'hint': 'T1cygGNXMoAfAAAARAAAAJoBAADAAAAAq6lSQQug30HSZipDACugQh8AAABEAAAAmgEAAMAAAAASAwAAL1A2B9lB3QBAUDYHoEDdABAAvxAAAAAA', 'location': [120.999983, 14.500313], 'name': '', 'distance': 34.68006026}, {'hint': '-d8lgD_gJYD2AAAACAAAAAAAAADRAAAA9qVNQitItz8AAAAAQdctQnsAAAAEAAAAAAAAAGgAAAASAwAArNY3Bx_H3gDg1jcHQMfeAAAALwkAAAAA', 'location': [121.099948, 14.599967], 'name': 'KC-30 Street', 'distance': 6.688139549}]}


## 2.1. Prepare School Coordinates for OSRM

In [19]:
display(df_pub.head(1))
display(df_priv.head(1))

,school_id,longitude,latitude,coord_valid,coord_missing,old_region,province
0,100001,120.614372,18.26686,True,False,Region I,ILOCOS NORTE


,school_id,longitude,latitude,coord_valid,old_region,province
0,400860,120.6149,17.58521,True,CAR,ABRA


In [20]:
# In 2.3b, '112354' was not found in the distance matrix of schools that is 8331 x 8331
# Let's check
sch_ids = ['112354','129283']
df_pub[df_pub['school_id'].isin(sch_ids)]

,school_id,longitude,latitude,coord_valid,coord_missing,old_region,province
15991,112354,114.003566,4.100519,False,False,Region V,CAMARINES SUR
37175,129283,126.079776,6.628162,True,False,Region XI,DAVAO ORIENTAL


In [21]:
# Combine public and private school coordinates (from Section 1)                                          
df_coords = pd.concat([df_pub, df_priv], ignore_index=True)                                               
print(df_coords.shape)

# Keep only valid coordinates                                                                             
df_coords = df_coords[df_coords['coord_valid'] == True].copy()                                            
                                                                                                        
# Filter to NCR, Region IV-A (CALABARZON), and Region III if needed                                                    
regions_of_interest = ['NCR', 'Region IV-A', 'Region III']                                                              
df_coords = df_coords[df_coords['old_region'].isin(regions_of_interest)].copy()                           
                                                                                                        
# Keep only required columns                                                                              
df_coords = df_coords[['school_id', 'longitude', 'latitude']].drop_duplicates()                           
df_coords = df_coords.reset_index(drop=True)                                                              
print(df_coords.shape)

print(f"Schools with valid coordinates: {len(df_coords):,}")                                              
print(f"\nSample:")                                                                                       
display(df_coords.head())

(59652, 7)
(13131, 3)
Schools with valid coordinates: 13,131

Sample:


,school_id,longitude,latitude
0,105692,121.037638,15.797337
1,105693,120.99518,15.85671
2,105694,120.99229,15.81548
3,105695,120.99469,15.78601
4,105696,121.023395,15.784624


## 2.2. Query ORSM

In [22]:
# OSRM service URL (within Docker network)                                                                
OSRM_URL = "http://osrm:5000/table/v1/driving/"                                                           
                                                                                                        
# Maximum coordinates per OSRM request (adjust if needed)                                                 
MAX_COORDS_PER_REQUEST = 500                                                                              
                                                                                                        
def get_osrm_distance_matrix(coords_list, batch_size=MAX_COORDS_PER_REQUEST):                             
  """                                                                                                   
  Get distance matrix from OSRM Table API.                                                              
                                                                                                        
  Args:                                                                                                 
      coords_list: List of (longitude, latitude) tuples                                                 
      batch_size: Max coordinates per request                                                           
                                                                                                        
  Returns:                                                                                              
      numpy array of distances in meters (np.inf where no route found)                                  
  """                                                                                                   
  n = len(coords_list)                                                                                  
                                                                                                        
  # If small enough, do single request                                                                  
  if n <= batch_size:                                                                                   
      coord_str = ";".join([f"{lon},{lat}" for lon, lat in coords_list])                                
      response = requests.get(                                                                          
          f"{OSRM_URL}{coord_str}",                                                                     
          params={"annotations": "distance"}                                                            
      )                                                                                                 
                                                                                                        
      if response.status_code != 200:                                                                   
          raise Exception(f"OSRM error: {response.text}")                                               
                                                                                                        
      data = response.json()                                                                            
      if data["code"] != "Ok":                                                                          
          raise Exception(f"OSRM error: {data.get('message', 'Unknown error')}")                        
                                                                                                        
      # Convert to numpy array, replace None with inf                                                   
      distances = np.array(data["distances"], dtype=np.float64)                                         
      distances[distances == None] = np.inf                                                             
                                                                                                        
      return distances                                                                                  
                                                                                                        
  # For larger datasets, we need to batch                                                               
  print(f"Dataset has {n} points, processing in batches...")                                            
                                                                                                        
  # Initialize full distance matrix                                                                     
  full_matrix = np.full((n, n), np.inf, dtype=np.float64)                                               
                                                                                                        
  # Process in batches (sources) against all destinations                                               
  for i in tqdm(range(0, n, batch_size), desc="Processing batches"):                                    
      batch_end = min(i + batch_size, n)                                                                
      batch_indices = list(range(i, batch_end))                                                         
                                                                                                        
      # Build coordinate string (all points)                                                            
      coord_str = ";".join([f"{lon},{lat}" for lon, lat in coords_list])                                
                                                                                                        
      # Specify which are sources (this batch) and destinations (all)                                   
      sources = ";".join(map(str, batch_indices))                                                       
                                                                                                        
      response = requests.get(                                                                          
          f"{OSRM_URL}{coord_str}",                                                                     
          params={                                                                                      
              "annotations": "distance",                                                                
              "sources": sources                                                                        
          }                                                                                             
      )                                                                                                 
                                                                                                        
      if response.status_code != 200:                                                                   
          print(f"Warning: Batch {i} failed with status {response.status_code}")                        
          continue                                                                                      
                                                                                                        
      data = response.json()                                                                            
      if data["code"] != "Ok":                                                                          
          print(f"Warning: Batch {i} failed - {data.get('message', 'Unknown error')}")                  
          continue                                                                                      
                                                                                                        
      # Fill in the distance matrix for this batch                                                      
      batch_distances = np.array(data["distances"], dtype=np.float64)                                   
      for j, src_idx in enumerate(batch_indices):                                                       
          full_matrix[src_idx, :] = batch_distances[j]                                                  
                                                                                                        
  return full_matrix                                                                                    
                                                                                                        
print("Distance matrix function ready.")

Distance matrix function ready.


## 2.3. Complete Dist Matrix

In [23]:
# Prepare coordinates as list of (lon, lat) tuples                                                        
coords_list = list(zip(df_coords['longitude'], df_coords['latitude']))                                    
school_ids = df_coords['school_id'].tolist()                                                              
                                                                                                        
print(f"Requesting distance matrix for {len(coords_list):,} schools...")                                  
print(f"This will compute {len(coords_list):,} x {len(coords_list):,} = {len(coords_list)**2:,} distances")                                                                                               
                                                                                                        
# Query OSRM                                                                                              
distance_matrix = get_osrm_distance_matrix(coords_list)                                                   
                                                                                                        
print(f"\nDistance matrix shape: {distance_matrix.shape}")                                                
print(f"Valid distances (not inf): {(distance_matrix < np.inf).sum():,}")                                 
print(f"Missing routes (inf): {(distance_matrix == np.inf).sum():,}")

Requesting distance matrix for 13,131 schools...
This will compute 13,131 x 13,131 = 172,423,161 distances
Dataset has 13131 points, processing in batches...


Processing batches: 100%|██████████| 27/27 [08:59<00:00, 20.00s/it]



Distance matrix shape: (13131, 13131)
Valid distances (not inf): 172,396,901
Missing routes (inf): 0


In [24]:
# Save the numpy matrix                                                                                                            
np.save('output/school_distance_matrix_osrm.npy', distance_matrix)                                                                 
                                                                                                                                 
# Create multi-value mapping: school_id → list of indices                                                                          
# This handles schools with multiple locations (ES vs JHS at different coordinates)                                                
school_id_to_indices = defaultdict(list)                                                                                           
for i, sid in enumerate(school_ids):                                                                                               
  school_id_to_indices[sid].append(i)                                                                                            
                                                                                                                                 
school_id_to_indices = dict(school_id_to_indices)                                                                                  
                                                                                                                                 
# Save index data                                                                                                                  
with open('output/school_distance_matrix_index.json', 'w') as f:                                                                   
  json.dump({                                                                                                                    
      'school_ids': school_ids,  # List preserving matrix order (8,331 entries)                                                  
      'school_id_to_indices': school_id_to_indices  # Dict of lists (8,307 unique IDs)                                           
  }, f, indent=2)                                                                                                                
                                                                                                                                 
# Summary                                                                                                                          
multi_location = sum(1 for v in school_id_to_indices.values() if len(v) > 1)                                                       
print(f"Saved distance matrix: {distance_matrix.shape}")                                                                           
print(f"School IDs in matrix: {len(school_ids):,}")                                                                                
print(f"Unique school IDs: {len(school_id_to_indices):,}")                                                                         
print(f"Schools with multiple locations: {multi_location}")                                                                        
print(f"Valid distances: {(distance_matrix < np.inf).sum():,}")                                                                    
print(f"Inf distances: {(distance_matrix == np.inf).sum():,}")

Saved distance matrix: (13131, 13131)
School IDs in matrix: 13,131
Unique school IDs: 13,106
Schools with multiple locations: 25
Valid distances: 172,396,901
Inf distances: 0


## 2.4. Inspect Dist Matrix

In [56]:
%%time
# =============================================================================                           
# Loading and Using the Distance Matrix                                                                   
# =============================================================================                           
# Load matrix and index
distance_matrix = np.load('output/school_distance_matrix_osrm.npy')

with open('output/school_distance_matrix_index.json', 'r') as f:                                          
    index_data = json.load(f)
    school_ids = index_data['school_ids']
    school_id_to_idx = index_data['school_id_to_indices']  # previously school_id_to_idx

CPU times: user 123 ms, sys: 1.38 s, total: 1.5 s
Wall time: 11.4 s


In [57]:
distance_matrix.shape

(13131, 13131)

In [58]:
len(school_ids), len(school_id_to_idx)

(13131, 13106)

In [59]:
# Check for duplicate school_ids in df_coords                                                                                      
duplicate_school_ids = schools_df[schools_df['school_id'].duplicated(keep=False)]                                                    
print(f"Rows with duplicate school_ids: {len(duplicate_school_ids)}")                                                              
print(f"Unique duplicate school_ids: {duplicate_school_ids['school_id'].nunique()}")                                               
display(duplicate_school_ids.sort_values('school_id')) 

Rows with duplicate school_ids: 50
Unique duplicate school_ids: 25


,school_id,longitude,latitude,sector
9889,401481,121.064430,14.866780,private
9890,401481,121.064776,14.862443,private
11646,401748,120.981790,14.397030,private
11560,401748,120.981314,14.396990,private
11562,401752,120.951300,14.437338,private
11647,401752,120.955831,14.436083,private
11649,401780,120.992346,14.392637,private
11567,401780,120.992297,14.392720,private
11643,401808,120.994420,14.400757,private
11572,401808,120.994559,14.400663,private


## 2.5. Verify Output

### Trim Dist Matrix
Code sourced from 2.4b notebook.

In [60]:
# Handle duplicate school IDs (keep first index, drop second)
# Some schools have 2 indices due to separate ES/JHS building coordinates
dupe_ids = [(schid, ids) for schid, ids in school_id_to_idx.items() if isinstance(ids, list) and len(ids) == 2]
idxs_to_drop = sorted([max(int(tup[1][0]), int(tup[1][1])) for tup in dupe_ids])
print(f"Duplicate IDs to resolve: {len(dupe_ids)}")

# Trim matrix by removing duplicate rows/columns
dm = distance_matrix.copy()
dm = np.delete(dm, idxs_to_drop, axis=1)
dm = np.delete(dm, idxs_to_drop, axis=0)
print(f"Trimmed distance matrix: {dm.shape}")

# Create old_idx -> new_idx mapping (accounting for dropped indices)
old_to_new_idx = {}
new_idx = 0
for old_idx in range(distance_matrix.shape[0]):
    if old_idx not in idxs_to_drop:
        old_to_new_idx[old_idx] = new_idx
        new_idx += 1

# Create school_id -> new_idx mapping
re_schids_to_idx = {}
for schid, idx_val in school_id_to_idx.items():
    # Get the first index (could be list or single value)
    if isinstance(idx_val, list):
        old_idx = min(int(v) for v in idx_val) # int(idx_val[0])  # Keep first index
    else:
        old_idx = int(idx_val)
    
    # Map to new index if it wasn't dropped
    if old_idx in old_to_new_idx:
        re_schids_to_idx[schid] = old_to_new_idx[old_idx]

re_schids = list(re_schids_to_idx.keys())
print(f"Schools in distance matrix: {len(re_schids):,}")

del distance_matrix  # Free memory

Duplicate IDs to resolve: 25
Trimmed distance matrix: (13106, 13106)
Schools in distance matrix: 13,106


### Save trimmed distance matrix

In [61]:
# Save the numpy matrix                                                                                                            
np.save('output/processed_school_distance_matrix_osrm.npy', dm)

# Save index data                                                                                                                  
with open('output/processed_school_distance_matrix_index.json', 'w') as f:                                                                   
  json.dump({                                                                                                                    
      'school_ids': re_schids,  # List preserving matrix order (13,106 entries)                                                  
      'school_id_to_indices': re_schids_to_idx  # Dict of lists (13,106 unique IDs)                                           
  }, f, indent=2)

### Check inter-school distances

In [20]:
display(sch_info.head(1))

,school_id,school_name,region,division,province,school_district,legislative_district,municipality,barangay,street_address,sector,school_management,annex_status,offers_es,offers_jhs,offers_shs,in_enr_2022_23,in_enr_2023_24,in_enr_2024_25,old_region
0,100001,Apaleng-Libtong ES,Region I,Ilocos Norte,ILOCOS NORTE,Bacarra I,1st District,BACARRA,LIBTONG,"Brgy. 21, Libtong, Bacarra, Ilocos Norte",Public,DepEd,Standalone School,True,False,False,True,True,True,Region I


In [19]:
display(dm.shape)

(13106, 13106)

In [18]:
display(schools_df.head(1))

,school_id,longitude,latitude,sector
0,105692,121.037638,15.797337,public


In [62]:
# Extract school info of Mega Manila regions (NCR, R4A, and R3)
rel_cols = ['school_id','school_name','old_region','division','sector']
mask = sch_info['old_region'].isin(['NCR', 'Region IV-A', 'Region III'])
mega_info = sch_info[rel_cols].loc[mask].copy()
print(mega_info.shape)

ncr_info = mega_info[mega_info['old_region'].isin(['NCR'])].copy()
r3_info = mega_info[mega_info['old_region'].isin(['Region III'])].copy()
r4a_info = mega_info[mega_info['old_region'].isin(['Region IV-A'])].copy()

print(f"Shape of ncr_info: {ncr_info.shape}")
print(f"Shape of r3_info: {r3_info.shape}")
print(f"Shape of r4a_info: {r4a_info.shape}")

(14377, 5)
Shape of ncr_info: (2842, 5)
Shape of r3_info: (5328, 5)
Shape of r4a_info: (6207, 5)


In [63]:
# NCR to R3
ncr_ids = ncr_info['school_id'].unique().tolist()
r3_ids = r3_info['school_id'].unique().tolist()

In [52]:
rand_ncr_schid = random.choice(ncr_ids)
ncr_dm_idx = re_schids_to_idx.get(rand_ncr_schid)
print(ncr_dm_idx)

rand_r3_schid = random.choice(r3_ids)
r3_dm_idx = re_schids_to_idx.get(rand_r3_schid)
print(r3_dm_idx)

9679
10113


In [53]:
# Display in school info df the two randomly chosen schools in NCR and R3
rand_info = mega_info[mega_info['school_id'].isin([rand_ncr_schid, rand_r3_schid])]
display(rand_info)

dist_in_dm = dm[np.ix_([ncr_dm_idx], [r3_dm_idx])]
display(dist_in_dm)

,school_id,school_name,old_region,division,sector
10497,401344,"Cabanatuan Adventist Elem School, Inc.",Region III,Cabanatuan City,Private
58895,406786,Holy Trinity School of St. Therese of the Chil...,NCR,Marikina City,Private


array([[136475.8]])

# 3.0. Scratch

## 3.1. Revise school_id_to_idx

In [29]:
# Load existing data                                                                                                               
distance_matrix = np.load('output/school_distance_matrix_osrm.npy')                                                                
                                                                                                                                 
with open('output/school_distance_matrix_index.json', 'r') as f:                                                                   
  index_data = json.load(f)                                                                                                      
  school_ids = index_data['school_ids']                                                                                          
  school_id_to_idx = index_data['school_id_to_idx']

print(f"Distance matrix shape: {distance_matrix.shape}")                                                                           
print(f"School IDs in list: {len(school_ids)}")    
print(f"Original length of school ID to index: {len(school_id_to_idx)}")
                                                                                                                                 
# Create multi-value mapping: school_id → list of indices                                                                          
school_id_to_indices = defaultdict(list)                                                                                           
for i, sid in enumerate(school_ids):                                                                                               
  school_id_to_indices[sid].append(i)                                                                                            
                                                                                                                                 
school_id_to_indices = dict(school_id_to_indices)

Distance matrix shape: (8331, 8331)
School IDs in list: 8331
Original length of school ID to index: 8307


In [31]:
# Verify                                                                                                                           
unique_ids = len(school_id_to_indices)                                                                                             
total_positions = sum(len(v) for v in school_id_to_indices.values())                                                               
multi_location_schools = {k: v for k, v in school_id_to_indices.items() if len(v) > 1}                                             
                                                                                                                                 
print(f"\nUnique school_ids: {unique_ids}")                                                                                        
print(f"Total matrix positions: {total_positions}")                                                                                
print(f"Schools with multiple locations: {len(multi_location_schools)}")                                                           
                                                                                                                                 
# Show schools with multiple locations                                                                                             
print(f"\nSchools with multiple coordinates:")                                                                                     
for sid, indices in multi_location_schools.items():                                                                                
  print(f"  {sid}: indices {indices}") 


Unique school_ids: 8307
Total matrix positions: 8331
Schools with multiple locations: 24

Schools with multiple coordinates:
  410951: indices [5267, 5312]
  401820: indices [6717, 6837]
  401861: indices [6722, 6841]
  409698: indices [6729, 6839]
  410285: indices [6731, 6840]
  424274: indices [6742, 6845]
  424275: indices [6743, 6838]
  424453: indices [6745, 6851]
  401748: indices [6760, 6846]
  401752: indices [6762, 6847]
  401780: indices [6767, 6849]
  401808: indices [6772, 6843]
  424327: indices [6779, 6822]
  408577: indices [6795, 6842]
  424060: indices [6802, 6848]
  424061: indices [6803, 6844]
  424152: indices [6805, 6852]
  424252: indices [6812, 6855]
  424318: indices [6818, 6854]
  424322: indices [6820, 6853]
  424380: indices [6825, 6850]
  402439: indices [7047, 7091]
  402444: indices [7048, 7092]
  410901: indices [7070, 7713]


In [32]:
# Structure: school_ids (list), school_id_to_indices (dict of lists)                                                                                                                                                                                            
updated_index_data = {                                                                                                             
  'school_ids': school_ids,  # Original list (8,331 entries, preserves order)                                                    
  'school_id_to_idx': school_id_to_indices  # New multi-value mapping                                                        
}                                                                                                                                  
                                                                                                                                 
# with open('output/school_distance_matrix_index.json', 'w') as f:                                                                   
#   json.dump(updated_index_data, f, indent=2)                                                                                     
                                                                                                                                 
print("Updated index file saved: output/school_distance_matrix_index.json")                                                        
print(f"  - school_ids: {len(school_ids)} entries (list)")                                                                         
print(f"  - school_id_to_indices: {len(school_id_to_indices)} unique school_ids (dict of lists)")

Updated index file saved: output/school_distance_matrix_index.json
  - school_ids: 8331 entries (list)
  - school_id_to_indices: 8307 unique school_ids (dict of lists)


In [44]:
dupe_sids = list(multi_location_schools.keys())

ess_cols = ['school_id','school_name','sector','offers_es','offers_jhs','offers_shs']
sch_info_dupes = sch_info[sch_info['school_id'].isin(dupe_sids)]
display(sch_info_dupes[ess_cols])

,school_id,school_name,sector,offers_es,offers_jhs,offers_shs
16796,401820,Seed of Wisdom Learning Center,Private,True,True,False
16801,401861,"Victorious Christian Montessori College, Inc.",Private,True,True,True
16808,409698,"PHILIPPINE SCHOOL OF ABUNDANT LIFE INC.,",Private,True,False,False
16810,410285,SECOND MOM LEARNING CENTER INC.,Private,True,False,False
16821,424274,"King James Academy, Cavite, Inc.",Private,True,True,False
16822,424275,Macasa Learning Center,Private,True,True,True
16824,424453,King Solomon Bacoor School Inc.,Private,True,True,False
16863,401748,Divine Light Academy,Private,True,True,True
16865,401752,"Emmaus Learning Center, Inc.",Private,True,True,False
16870,401780,Berrien Springs Academy,Private,True,False,False


## 3.2. Inpsect update

### Helper

In [34]:
  def get_distance(origin_id, dest_id, method='min'):                                                                                
      """                                                                                                                            
      Get distance between two schools.                                                                                              
                                                                                                                                     
      Args:                                                                                                                          
          origin_id: Origin school ID                                                                                                
          dest_id: Destination school ID                                                                                             
          method: How to handle schools with multiple locations                                                                      
                  'min' (default) - shortest distance between any location pair                                                      
                  'max' - longest distance                                                                                           
                  'mean' - average distance                                                                                          
                                                                                                                                     
      Returns:                                                                                                                       
          Distance in meters, or np.inf if no route found                                                                            
      """                                                                                                                            
      origin_indices = school_id_to_indices.get(str(origin_id), [])                                                                  
      dest_indices = school_id_to_indices.get(str(dest_id), [])                                                                      
                                                                                                                                     
      if not origin_indices or not dest_indices:                                                                                     
          return np.inf                                                                                                              
                                                                                                                                     
      # Get all pairwise distances                                                                                                   
      distances = []                                                                                                                 
      for i in origin_indices:                                                                                                       
          for j in dest_indices:                                                                                                     
              d = distance_matrix[i, j]                                                                                              
              if d < np.inf:                                                                                                         
                  distances.append(d)                                                                                                
                                                                                                                                     
      if not distances:                                                                                                              
          return np.inf                                                                                                              
                                                                                                                                     
      if method == 'min':                                                                                                            
          return min(distances)                                                                                                      
      elif method == 'max':                                                                                                          
          return max(distances)                                                                                                      
      else:  # mean                                                                                                                  
          return np.mean(distances)                                                                                                  
                                                                                                                                     
                                                                                                                                     
  def get_nearby_schools(school_id, max_distance_m=5000, method='min'):                                                              
      """                                                                                                                            
      Get all schools within X meters of a school.                                                                                   
                                                                                                                                     
      For schools with multiple locations, uses the specified method                                                                 
      to determine the effective distance.                                                                                           
      """                                                                                                                            
      origin_indices = school_id_to_indices.get(str(school_id), [])                                                                  
                                                                                                                                     
      if not origin_indices:                                                                                                         
          return []                                                                                                                  
                                                                                                                                     
      nearby = {}                                                                                                                    
      for target_id, target_indices in school_id_to_indices.items():                                                                 
          if target_id == str(school_id):                                                                                            
              continue                                                                                                               
                                                                                                                                     
          # Get distances from all origin locations to all target locations                                                          
          distances = []                                                                                                             
          for i in origin_indices:                                                                                                   
              for j in target_indices:                                                                                               
                  d = distance_matrix[i, j]                                                                                          
                  if d < np.inf:                                                                                                     
                      distances.append(d)                                                                                            
                                                                                                                                     
          if distances:                                                                                                              
              if method == 'min':                                                                                                    
                  effective_dist = min(distances)                                                                                    
              elif method == 'max':                                                                                                  
                  effective_dist = max(distances)                                                                                    
              else:                                                                                                                  
                  effective_dist = np.mean(distances)                                                                                
                                                                                                                                     
              if 0 < effective_dist <= max_distance_m:                                                                               
                  nearby[target_id] = effective_dist                                                                                 
                                                                                                                                     
      # Sort by distance                                                                                                             
      return sorted(nearby.items(), key=lambda x: x[1])

### Implem

In [35]:
# # Test the updated functions                                                                                                       
# print("=== Testing Updated Functions ===\n")                                                                                       
                                                                                                                                 
# Test with a school that has multiple locations                                                                                   
test_school = list(multi_location_schools.keys())[0]                                                                               
print(f"Test school {test_school} has indices: {school_id_to_indices[test_school]}")

Test school 410951 has indices: [5267, 5312]


In [40]:
type(school_id_to_indices)

dict

In [39]:
display(school_id_to_indices.get('410951'))

[5267, 5312]

In [38]:
# Get distance to another school                                                                                                   
other_school = [sid for sid in school_id_to_indices.keys() if sid != test_school][0]                                               
dist_min = get_distance(test_school, other_school, method='min')                                                                   
dist_max = get_distance(test_school, other_school, method='max')                                                                   
print(f"\nDistance from {test_school} to {other_school}:")                                                                         
print(f"  Min: {dist_min/1000:.4f} km")                                                                                            
print(f"  Max: {dist_max/1000:.4f} km")


Distance from 410951 to 107900:
  Min: 31.1727 km
  Max: 31.1741 km


In [37]:
# Get nearby schools                                                                                                               
nearby = get_nearby_schools(test_school, max_distance_m=3000)                                                                      
print(f"\nSchools within 3km of {test_school}: {len(nearby)}")                                                                     
if nearby[:3]:                                                                                                                     
  print("  Top 3 nearest:")                                                                                                      
  for sid, dist in nearby[:3]:                                                                                                   
      print(f"    {sid}: {dist/1000:.2f} km")


Schools within 3km of 410951: 45
  Top 3 nearest:
    407673: 0.17 km
    407055: 0.24 km
    407048: 0.30 km
